# Vibrational modes from an optimized molecule

This local notebook optimizes a molecule with xTB through VeloxChem, calculates a numerical Hessian, obtains the normal vibrational modes, and animates the modes with `py3Dmol`.

Use small or medium-sized molecules first. Vibrational calculations require many gradient evaluations and can become slow for large molecules.

In [ ]:
# Local environment check for QuimicaAutomocio P2.
# The original eChem notebooks are Colab-oriented; this one is designed for local use.
import importlib
import numpy as np
import py3Dmol
import veloxchem as vlx

required = ["XtbDriver", "OptimizationDriver", "VibrationalAnalysis"]
missing = [name for name in required if not hasattr(vlx, name)]
if missing:
    raise RuntimeError(f"VeloxChem is missing required classes: {missing}")

# If xtb-python was installed after the kernel started, restart the kernel.
import xtb  # noqa: F401

print("VeloxChem", getattr(vlx, "__version__", "unknown"))
print("Environment OK")

## 1. Define the molecule

Choose either a SMILES string or explicit XYZ coordinates. The default example is water because it is fast and has three normal vibrational modes.

In [ ]:
use_smiles = True

# Try: "O" water, "CO" methanol, "CCO" ethanol, "C=O" formaldehyde,
# "CC(=O)C" acetone, "c1ccccc1" benzene.
smiles = "O"

xyz_coordinates = """
O  0.000000  0.000000  0.000000
H  0.758602  0.000000  0.504284
H -0.758602  0.000000  0.504284
"""

if use_smiles:
    molecule = vlx.Molecule.read_smiles(smiles)
else:
    molecule = vlx.Molecule.read_str(xyz_coordinates)

print(molecule.get_xyz_string())
molecule.show(atom_indices=True, width=520, height=360)

## 2. Optimize the geometry

The optimization uses xTB. This is much faster than DFT and is adequate for a teaching workflow where the goal is to inspect the character of the normal modes.

In [ ]:
xtb_drv = vlx.XtbDriver()
xtb_drv.ostream.mute()

opt_drv = vlx.OptimizationDriver(xtb_drv)
opt_drv.ostream.mute()
opt_drv.max_iter = 100

opt_results = opt_drv.compute(molecule)
opt_molecule = opt_results["final_molecule"]

print("Optimized geometry:")
print(opt_molecule.get_xyz_string())
print("Optimization steps:", len(opt_results.get("opt_energies", [])))
opt_molecule.show(atom_indices=True, width=520, height=360)

## 3. Calculate the Hessian and vibrational frequencies

A true minimum should not have imaginary frequencies. In practice, small negative values can appear if the geometry is not fully optimized or if the numerical Hessian is noisy.

In [ ]:
vib_drv = vlx.VibrationalAnalysis(xtb_drv)
vib_drv.ostream.mute()
vib_drv.do_ir = True

vib_results = vib_drv.compute(opt_molecule)

frequencies = np.array(vib_results["vib_frequencies"], dtype=float)
normal_modes = np.array(vib_results["normal_modes"], dtype=float)
reduced_masses = np.array(vib_results["reduced_masses"], dtype=float)
force_constants = np.array(vib_results["force_constants"], dtype=float)
ir_intensities = np.array(vib_results.get("ir_intensities", np.zeros_like(frequencies)), dtype=float)

print("mode  frequency / cm^-1   red. mass / amu   force const. / mdyn A^-1   IR / km mol^-1")
for i, freq in enumerate(frequencies, start=1):
    print(f"{i:>4d}  {freq:>16.2f}  {reduced_masses[i-1]:>15.4f}  {force_constants[i-1]:>25.4f}  {ir_intensities[i-1]:>14.2f}")

## 4. Animate one normal mode

Select the mode number in the next cell. The animation is generated as a short XYZ trajectory by displacing the optimized geometry along the selected normal coordinate.

In [ ]:
def normal_mode_trajectory(molecule, mode, amplitude=0.45, frames=32):
    labels = list(molecule.get_labels())
    coords = np.array(molecule.get_coordinates_in_angstrom(), dtype=float)
    mode = np.array(mode, dtype=float)

    chunks = []
    for phase in np.linspace(0.0, 2.0 * np.pi, frames, endpoint=False):
        displaced = coords + amplitude * np.sin(phase) * mode
        chunks.append(str(len(labels)))
        chunks.append("normal-mode frame")
        for label, xyz in zip(labels, displaced):
            chunks.append(f"{label:2s} {xyz[0]: .8f} {xyz[1]: .8f} {xyz[2]: .8f}")
    return "\n".join(chunks) + "\n"

def show_mode(molecule, mode_index, amplitude=0.45, frames=32):
    traj = normal_mode_trajectory(molecule, normal_modes[mode_index], amplitude=amplitude, frames=frames)
    view = py3Dmol.view(width=650, height=430)
    view.addModelsAsFrames(traj, "xyz")
    view.setStyle({"stick": {"radius": 0.16}, "sphere": {"scale": 0.24}})
    view.animate({"loop": "backAndForth", "reps": 0, "interval": 80})
    view.zoomTo()
    return view

# Python index: 0 is the first vibrational mode. Change this value.
mode_index = 0
print(f"Showing mode {mode_index + 1}: {frequencies[mode_index]:.2f} cm^-1")
show_mode(opt_molecule, mode_index, amplitude=0.45, frames=32).show()

## 5. Questions for the report

1. Which normal mode has the highest frequency? Is it mainly a bond stretch, a bend, or a collective motion?
2. Compare two modes: which one should have the larger force constant, and why?
3. If the molecule contains O-H, N-H or C-H bonds, compare their stretching frequencies with the qualitative expectations from bond strength and reduced mass.
4. Explain why the calculation must be done on an optimized geometry before the vibrational analysis.